# 1.1 — Snowflake Gen AI Principles & Features

**Exam domain:** Domain 1.0 — Snowflake Gen AI Overview · **Weight:** 19%

## The problem this solves

Your support team closes a few thousand tickets a month. Someone asks for a weekly breakdown of what customers are actually complaining about. The tickets are free text, they arrive in several languages, and the answer changes every week. The usual route is to export the text somewhere else, run a model on it, and bring the results back — which means a copy of customer data now lives outside the systems your security team signed off on.

The point of this notebook is that you do not have to move the data. You can call a language model from the same SQL that already reads the table.

## What you will be able to do

- Name each Snowflake Cortex capability and say which problem it is for
- Check whether your role is actually allowed to call an AI function, and fix it if not
- Call `AI_COMPLETE` and read its detailed output
- Decide between fine-tuning and retrieval when a model gives you the wrong answers
- Control which regions your inference requests are allowed to reach

## Before you start

- Run `setup/dataset.sql` once. It creates the `GENAI_STUDY` database and the tables these cells read.
- A running warehouse. Cortex AI functions are charged against the warehouse that runs the query.
- A role that can reach the AI functions. Section 4 shows you how to check; on most trial accounts the default grants are already enough.

📖 **Snowflake documentation for this notebook**
- [Snowflake Cortex AISQL](https://docs.snowflake.com/en/user-guide/snowflake-cortex/aisql)
- [Cortex AI functions: privileges and access](https://docs.snowflake.com/en/user-guide/snowflake-cortex/aisql-privileges-and-access)
- [SHOW CORTEX BASE MODELS](https://docs.snowflake.com/en/sql-reference/sql/show-cortex-base-models)
- [Cortex Fine-tuning](https://docs.snowflake.com/en/user-guide/snowflake-cortex/cortex-finetuning)
- [Cross-region inference](https://docs.snowflake.com/en/user-guide/snowflake-cortex/cross-region-inference)
- [Snowflake database roles](https://docs.snowflake.com/en/sql-reference/snowflake-db-roles)

---
## 1. What Snowflake Cortex is

**Snowflake Cortex** is the set of AI features that run inside your Snowflake account. A *warehouse* — the compute cluster that executes your SQL — submits the request, the inference happens inside Snowflake's own service boundary, and the answer comes back as a column. You do not stand up an endpoint and you do not hand the text to a third party you have to contract with separately.

Seven things live under that name, and they solve different problems. The most common exam mistake is not knowing a definition — it is naming the wrong one of these for a given scenario.

| Capability | The problem it solves |
|---|---|
| **Cortex AI functions (AISQL)** | You want a model's answer as a column. `AI_COMPLETE`, `AI_CLASSIFY`, `AI_EXTRACT`, `AI_SENTIMENT`, `AI_TRANSLATE`, `AI_FILTER`, `AI_AGG` and friends |
| **Cortex Search** | You have documents, and you need the handful of passages relevant to a question. Hybrid lexical + vector retrieval with reranking |
| **Cortex Analyst** | Business users want numbers out of *tables* without writing SQL. Text-to-SQL over a semantic view |
| **Cortex Agents** | One question needs both documents and tables, and several steps. A managed plan → use tools → reflect loop |
| **Cortex Fine-tuning** | A base model keeps answering in the wrong style or vocabulary and prompting is not fixing it |
| **Cortex Code (CoCo)** | You want AI help writing the SQL and Python itself, in Snowsight, a desktop app, or a terminal |
| **Model Registry / SPCS** | The model is *yours*, not Snowflake's, and you need somewhere governed to run it |

A *semantic view* is a schema-level object that tells Cortex Analyst which tables are facts, which columns are dimensions, and what the business calls each one. → [More on Cortex Analyst](https://docs.snowflake.com/en/user-guide/snowflake-cortex/cortex-analyst)

> **On names:** Snowflake's ready-made conversational app for business users is currently documented as **Snowflake CoWork**. Older material — including the published exam study guide — calls the same thing *Snowflake CoWork (the feature Snowflake previously called Snowflake Intelligence; the exam study guide may still use the older name)*. Both names describe an application built on Cortex Agents; neither is a separate retrieval capability.

---
## 2. Bringing your own model

Not everything is a hosted LLM. If you trained a model yourself, there are two governed places to put it.

| Path | Use it when |
|---|---|
| **Snowflake Model Registry** | You have a serializable Python model object — sklearn, XGBoost, PyTorch, HuggingFace. Log it once, then serve it from a warehouse or from containers |
| **Snowpark Container Services (SPCS)** | You need an arbitrary container: a custom runtime, a GPU, or a long-running server |

The registry decides where a version can be served with `target_platforms`, which takes `WAREHOUSE`, `SNOWPARK_CONTAINER_SERVICES`, or both. Warehouse-served models are capped at 15 GB. → [More on the Model Registry](https://docs.snowflake.com/en/developer-guide/snowflake-ml/model-registry/overview)

Both paths get a full deep dive later: [DD1 — Snowpark Container Services](../Deep%20Dives/DD1%20-%20Snowpark%20Container%20Services.ipynb) and [DD2 — Model Registry](../Deep%20Dives/DD2%20-%20Model%20Registry.ipynb).

---
## 3. How you reach Cortex

| Interface | Looks like | Worth knowing |
|---|---|---|
| **SQL** | `SELECT AI_COMPLETE(...)` | The canonical surface. Works anywhere SQL runs |
| **Snowpark Python** | `from snowflake.snowpark.functions import ai_complete` | The same functions, as DataFrame expressions |
| **REST — inference** | `POST /api/v2/cortex/inference:embed` | For apps outside Snowflake |
| **REST — Analyst** | `POST /api/v2/cortex/analyst/message` | Body carries `messages` plus one of `semantic_view` / `semantic_model_file` |
| **Snowsight** | Cortex Playground, agent builder, semantic view Autopilot | No-code, good for trying a model before you write SQL |

The older `SNOWFLAKE.CORTEX.COMPLETE` function and the `snowflake.cortex` Python wrappers still work, but the reference page for `SNOWFLAKE.CORTEX.COMPLETE` states that this legacy function **will be deprecated by the end of 2026** and points at `AI_COMPLETE` as the replacement. Write new code against `AI_*`. → [More on the legacy COMPLETE function](https://docs.snowflake.com/en/sql-reference/functions/complete-snowflake-cortex)

---
## 4. Who is allowed to call an AI function

This is a two-part rule, and missing either half produces the same unhelpful permission error.

A caller needs **both**:

1. The account-level privilege **`USE AI FUNCTIONS`** — or a per-function grant such as `USE AI FUNCTION AI_COMPLETE`, and
2. **one of** the database roles `SNOWFLAKE.CORTEX_USER` or `SNOWFLAKE.AI_FUNCTIONS_USER`.

A *database role* is a role that exists inside a specific database — here, the `SNOWFLAKE` database — and has to be granted to an account role before anyone can use it.

| Database role | What it covers | Granted to PUBLIC by default? |
|---|---|---|
| `SNOWFLAKE.CORTEX_USER` | Cortex features broadly | **Yes** |
| `SNOWFLAKE.AI_FUNCTIONS_USER` | Scalar AI functions only — every Cortex AI function *except* the aggregates `AI_AGG` and `AI_SUMMARIZE_AGG` | No |
| `SNOWFLAKE.CORTEX_EMBED_USER` | `AI_EMBED`, `EMBED_TEXT_768`, `EMBED_TEXT_1024` | No |
| `SNOWFLAKE.CORTEX_ANALYST_USER` | Cortex Analyst only | No |
| `SNOWFLAKE.CORTEX_AGENT_USER` | The Cortex Agents API only | No |
| `SNOWFLAKE.CORTEX_REST_API_USER` | The Cortex REST API only | No |
| `SNOWFLAKE.COPILOT_USER` | Cortex Code features in Snowsight | **Yes** |

The narrow roles exist so that you can give a team exactly one capability. Granting `CORTEX_ANALYST_USER` lets a business analyst ask questions of a semantic view without also handing them `AI_COMPLETE` against every model in the account.

To grant a single function rather than all of them:

```sql
GRANT USE AI FUNCTION AI_COMPLETE ON ACCOUNT TO ROLE ai_complete_user_role;
```

That per-function privilege stands **in place of** the blanket `USE AI FUNCTIONS`; you still need one of the database roles alongside it.

→ [More on Cortex privileges and access](https://docs.snowflake.com/en/user-guide/snowflake-cortex/aisql-privileges-and-access)

---
## 5. Where governance actually bites

- Inference runs inside Snowflake's boundary. You are not managing an external endpoint, so there is no separate data-processing agreement to negotiate for it.
- Which *models* your account can reach depends on your region, and on whether you allow requests to leave it — see section 8.
- Access is layered: account privilege → database role → the specific model → ordinary object grants on the tables you are reading. A permission failure can come from any of the four layers, so check them in that order.

> ### ⚠️ Common misconceptions
>
> **"I have `SNOWFLAKE.CORTEX_USER`, so I can call any AI function."**
> The database role is only half of the rule. Without the `USE AI FUNCTIONS` account privilege — or a per-function `USE AI FUNCTION <name>` grant — the call fails on privileges, not on the model. The error does not tell you which of the two halves is missing, so check both.
> → [Cortex privileges and access](https://docs.snowflake.com/en/user-guide/snowflake-cortex/aisql-privileges-and-access)
>
> **"`AI_FUNCTIONS_USER` is just a smaller version of `CORTEX_USER`, so aggregates work too."**
> It covers the scalar functions only. `AI_AGG` and `AI_SUMMARIZE_AGG` are excluded by design, so a pipeline that classifies rows fine will suddenly fail the moment someone adds a summarise-per-group step. That is the exact failure this role is meant to cause.
> → [Cortex privileges and access](https://docs.snowflake.com/en/user-guide/snowflake-cortex/aisql-privileges-and-access)
>
> **"There must be a view in `INFORMATION_SCHEMA` listing the models I can use."**
> The documented way to list base models is the command `SHOW CORTEX BASE MODELS`, reading from the `SNOWFLAKE.MODELS` schema. Snowflake recommends adding `IN SCHEMA SNOWFLAKE.MODELS` — without it the command follows normal scoping rules and can return **zero rows** even when your grants are correct, which reads like "no models available" rather than "wrong scope".
> → [SHOW CORTEX BASE MODELS](https://docs.snowflake.com/en/sql-reference/sql/show-cortex-base-models)

---
## 6. Fine-tuning, and when it is the wrong tool

Reach for fine-tuning when the model's *behaviour* is wrong: it answers in the wrong register, ignores your house format, or does not know your internal vocabulary. Do not reach for it when the model's *facts* are wrong — new facts belong in retrieval, because a fine-tuned model still has a fixed training cut-off and you would have to re-tune every time the facts change.

### The function

`SNOWFLAKE.CORTEX.FINETUNE` is one function with four commands, and the arguments differ per command.

```sql
SELECT SNOWFLAKE.CORTEX.FINETUNE(
    'CREATE',                                        -- command
    '<db>.<schema>.<model_name>',                    -- name of the model to create
    'llama3.1-8b',                                   -- base model
    'SELECT prompt, completion FROM <training_tbl>', -- training query
    'SELECT prompt, completion FROM <val_tbl>'       -- optional validation query
) AS job_id;
```

| Command | Arguments | Returns |
|---|---|---|
| `'CREATE'` | model name, base model, training query, optional validation query | a job id |
| `'SHOW'` | none | every fine-tuning job in the account |
| `'DESCRIBE'` | the **job id** | progress and status |
| `'CANCEL'` | the **job id** | cancels the job |

### The details that decide whether it runs

- **Base model:** `llama3.1-8b` is the model documented as available for fine-tuning, with a 24k-token context window — 20k of prompt and 4k of completion.
- **Training data:** the query must return columns literally named **`prompt`** and **`completion`**. Alias them if your table calls them something else. Other columns are ignored.
- **Privileges:** `USAGE` on the database holding the training data, and `CREATE MODEL` (or ownership) on the schema where the model is saved. ACCOUNTADMIN must have granted `SNOWFLAKE.CORTEX_USER` to the calling role.
- **Calling the result:** it becomes a model name like any other — `SELECT AI_COMPLETE('<db>.<schema>.<model_name>', 'prompt text');`

### What it costs you

A fine-tuned model is a new object you now own: someone has to re-tune it when the data drifts, and the tuning job itself consumes credits before you know whether it helped. Retrieval costs you an indexing pipeline instead, but the content stays editable. The honest rule is to try prompting first, retrieval second, and fine-tuning when you have labelled examples and a format requirement that neither of the first two is meeting.

→ [More on Cortex Fine-tuning](https://docs.snowflake.com/en/user-guide/snowflake-cortex/cortex-finetuning)

In [ ]:
%%sql -r cortex_user_grants
-- Example 1: what does CORTEX_USER actually carry? Start here when a call fails on privileges.
SHOW GRANTS TO DATABASE ROLE SNOWFLAKE.CORTEX_USER;

In [ ]:
%%sql -r available_models
-- Example 2: which base models can THIS role use, and in which regions?
-- Always qualify with IN SCHEMA SNOWFLAKE.MODELS. Snowflake recommends it because without
-- the clause the command follows normal scoping rules and can return zero rows.
-- The result is filtered by the model grants your role holds.
SHOW CORTEX BASE MODELS IN SCHEMA SNOWFLAKE.MODELS;

-- Columns worth reading: name, lifecycle_status (GA / PUPR / PRPR / LEGACY / EOL),
--                        in_region_availability, cross_region_availability,
--                        legacy_date, eol_date
--
-- SNOWFLAKE.MODELS is refreshed daily by a Snowflake-managed task. ACCOUNTADMIN can pull
-- a newly released model in early with:
--   CALL SNOWFLAKE.MODELS.CORTEX_BASE_MODELS_REFRESH();

In [ ]:
%%sql -r basic_complete
-- Example 3: the smallest possible AI_COMPLETE call. Run this first to prove the
-- privilege chain and the region routing both work before you debug anything else.
-- Do not hard-code a model name from a blog post: check SHOW CORTEX BASE MODELS for
-- what your account can actually reach and what its lifecycle_status is.
SELECT AI_COMPLETE(
    'llama3.1-8b',
    'In one sentence, what is Snowflake Cortex?'
) AS response;

In [ ]:
%%sql -r complete_detailed
-- Example 4: the same call with hyperparameters and inference metadata.
-- model_parameters and show_details are SEPARATE arguments, not keys of one options object.
SELECT
    AI_COMPLETE(
        model => 'llama3.1-8b',
        prompt => 'Summarize: Snowflake Cortex runs LLM inference inside the account.',
        model_parameters => {
            'temperature': 0,     -- 0 to 1. Default 0: repeatable output
            'max_tokens': 4096,   -- default 4096. Caps the OUTPUT only
            'guardrails': TRUE    -- default FALSE. Filters unsafe responses
        },
        show_details => TRUE      -- adds choices / created / model / usage token counts
    ) AS detailed;

-- Snowpark Python equivalent:
--   from snowflake.snowpark.functions import ai_complete
--   df.select(ai_complete('llama3.1-8b', col('ticket_text')))

In [ ]:
%%sql -r fine_tune_jobs
-- Example 5: list every fine-tuning job in the account. 'SHOW' takes no other argument.
SELECT SNOWFLAKE.CORTEX.FINETUNE('SHOW') AS fine_tune_jobs;

-- DESCRIBE and CANCEL take the JOB ID returned by 'CREATE', not the model name:
--   SELECT SNOWFLAKE.CORTEX.FINETUNE('DESCRIBE', '<job_id>');
--   SELECT SNOWFLAKE.CORTEX.FINETUNE('CANCEL',   '<job_id>');

> ### 🤔 Stop and think
>
> - Your account has `CORTEX_USER` granted to `PUBLIC`, which is the default. Every user in the account can therefore spend credits on model calls. What would you actually change, and what does the stricter setup cost you in support tickets from people who can no longer run their queries?
> - You can fine-tune a model, or you can index the same content for retrieval. Six months from now, which of those two is cheaper to keep correct — and who on your team owns that work?
> - `temperature: 0` makes output repeatable, which makes it testable. When would you deliberately give that up?

---
## 7. Scenario — the model is not available in your region

**Situation:** your team processes support tickets in English and Spanish from an account in AWS us-east-1. The model you want to use is not deployed there.

**Q1:** What controls whether Snowflake may send the request to a model in another region?

**Q2:** What does the calling role need, and who grants it?

### Worked solution

**A1:** The account-level parameter `CORTEX_ENABLED_CROSS_REGION`. It is not a per-query setting and not a warehouse setting — it is set once for the account, and only `ACCOUNTADMIN` can set it. `ORGADMIN` cannot.

```sql
ALTER ACCOUNT SET CORTEX_ENABLED_CROSS_REGION = 'ANY_REGION';             -- any region, any cloud
ALTER ACCOUNT SET CORTEX_ENABLED_CROSS_REGION = 'AWS_GLOBAL';             -- any AWS region
ALTER ACCOUNT SET CORTEX_ENABLED_CROSS_REGION = 'AWS_US';                 -- AWS US regions only
ALTER ACCOUNT SET CORTEX_ENABLED_CROSS_REGION = 'AWS_GLOBAL,AZURE_GLOBAL';-- combinations allowed
ALTER ACCOUNT SET CORTEX_ENABLED_CROSS_REGION = 'DISABLED';               -- home region only
```

**Documented values:**

| Tier | Values |
|---|---|
| Everything | `ANY_REGION` |
| Whole cloud | `AWS_GLOBAL` · `AZURE_GLOBAL` · `GCP_GLOBAL` |
| Cloud + geography | `AWS_US` · `AWS_EU` · `AWS_APJ` · `AWS_JP` · `AWS_AU` · `AZURE_US` · `AZURE_EU` · `GCP_US` |
| Nothing | `DISABLED` |

Comma-separated combinations of the cloud-specific and geography values are allowed. New organizations created after **9 March 2026** use `ANY_REGION`; other commercial accounts inherit a same-cloud geography value such as `AWS_US` or `AZURE_EU`.

The granularity stops at cloud plus geography. There is no value naming a single physical region, so `'us-east-1'` is not a setting — and neither are invented forms like `WITHIN_AWS`.

**A2:** The role needs `USE AI FUNCTIONS ON ACCOUNT` **and** one of `SNOWFLAKE.CORTEX_USER` or `SNOWFLAKE.AI_FUNCTIONS_USER`. `CORTEX_USER` is granted to `PUBLIC` by default; granting or revoking these requires `ACCOUNTADMIN`.

```sql
GRANT DATABASE ROLE SNOWFLAKE.CORTEX_USER TO ROLE MY_ANALYST_ROLE;
GRANT USE AI FUNCTIONS ON ACCOUNT TO ROLE MY_ANALYST_ROLE;
```

→ [More on cross-region inference](https://docs.snowflake.com/en/user-guide/snowflake-cortex/cross-region-inference)

> ### ⚠️ Common misconceptions
>
> **"Turning on cross-region inference means my data leaves the private network and gets stored somewhere else."**
> Within one cloud provider the traffic stays on that provider's private backbone. Across providers it crosses the public internet protected by mutual TLS. In neither case is customer data stored at the processing region. Refusing cross-region on this reasoning costs you model availability for a risk the documentation says is already handled.
> → [Cross-region inference](https://docs.snowflake.com/en/user-guide/snowflake-cortex/cross-region-inference)
>
> **"If the request is processed in another region, that region's account gets billed and I pay egress."**
> Credits are consumed in your requesting region regardless of where the inference runs, and cross-region inference does not incur data egress charges. The cost surprise people expect here does not happen; the real cost surprise is latency.
> → [Cross-region inference](https://docs.snowflake.com/en/user-guide/snowflake-cortex/cross-region-inference)

In [ ]:
%%sql -r cross_region_param
-- Scenario check: what is this account's cross-region setting right now?
SHOW PARAMETERS LIKE 'CORTEX_ENABLED_CROSS_REGION' IN ACCOUNT;

---
## 8. Cortex Code and Copilot — the AI that writes the SQL

Everything above is AI applied to your data. This section is AI applied to *you*: assistance while authoring the SQL and Python itself. There is nothing to run from a notebook, so learn it as recall.

### Cortex Code (CoCo) — three surfaces

| Surface | What it is |
|---|---|
| **CoCo in Snowsight** | Integrated into Workspaces and the Snowsight admin pages. SQL and Python notebook authoring, account administration, workspace-aware context, and a visual diff of suggested changes before you apply them |
| **CoCo Desktop** | A native macOS/Windows application with the full agent experience — agent and editor modes, local file access for projects such as dbt and Streamlit, and extensibility through plugins, skills and hooks |
| **CoCo CLI** | A terminal client that connects your local environment to a Snowflake account, with bash, git and SQL tool orchestration |

All three are generally available and require cross-region inference to be enabled.

### CoCo CLI — the flags worth remembering

```bash
cortex -c production          # -c / --connection : named Snowflake connection
cortex -w /path/to/project    # -w / --workdir    : working directory for file operations
cortex -m <model_name>        # -m / --model      : choose the model
cortex --continue             # resume the MOST RECENT conversation
cortex -r <session_id>        # -r / --resume     : resume a SPECIFIC session, or 'last'
cortex -p "<prompt>"          # -p / --print      : run one prompt, print, exit
cortex --plan                 # require approval before all actions
cortex --bypass               # approve all tool calls automatically
cortex --private              # do not save this session to history
cortex --cloud                # run tools in a managed container
```

Inside a running session, `/resume` (aliases `/r`, `/sessions`) lists sessions, `/fork` branches the current one, and `/rewind` steps back. The split to remember: **`--continue` takes no argument and resumes the most recent conversation; `-r` / `--resume` takes a session id** (or the literal `last`).

→ [More on the CoCo CLI](https://docs.snowflake.com/en/user-guide/cortex-code/cli-reference)

### Snowflake Copilot

- An LLM-powered assistant inside Snowsight for exploring data, generating SQL from a question, improving queries and fixing syntax errors. It respects the caller's role-based access.
- **Role:** `SNOWFLAKE.COPILOT_USER`, granted to `PUBLIC` by default.
- **Models:** Claude Sonnet 3.5 in AWS US West, AWS US East, and regions with cross-region inference enabled, falling back to Mistral Large 2. Nine regions have native support; anywhere else depends on `CORTEX_ENABLED_CROSS_REGION`.
- **Disable it** by revoking the role: `REVOKE DATABASE ROLE SNOWFLAKE.COPILOT_USER FROM ROLE PUBLIC;`
- Currently free to use.

→ [More on Snowflake Copilot](https://docs.snowflake.com/en/user-guide/snowflake-copilot)

In [ ]:
%%sql -r copilot_grants
-- Copilot access check: this role must reach the caller's role for the assistant to appear
SHOW GRANTS TO DATABASE ROLE SNOWFLAKE.COPILOT_USER;

---
## 9. Cortex Search, RAG, and unstructured data

### What "unstructured" means here

Text with no schema: support emails, contracts, PDFs, call transcripts, knowledge-base articles, product reviews. The route into Snowflake is the same three steps for all of them — **get it into text → chunk it → index it**:

```
files on a stage
   → AI_PARSE_DOCUMENT / AI_TRANSCRIBE       documents, audio, video → text
   → SPLIT_TEXT_RECURSIVE_CHARACTER          cut into retrievable chunks
   → CREATE CORTEX SEARCH SERVICE            lexical index + vector index, managed
```

A *stage* is a named location Snowflake can read files from — internal storage or your own cloud bucket.

### The RAG loop, and which piece does what

| Stage | The Snowflake piece |
|---|---|
| **Retrieve** | Cortex Search — lexical and vector retrieval combined, then semantic reranking |
| **Augment** | put the retrieved chunks into the prompt as context |
| **Generate** | `AI_COMPLETE`, instructed to answer only from that context |
| **Evaluate** | AI Observability — context relevance, groundedness, answer relevance, coherence, correctness |

Cortex Search is the retrieval half. It does not generate text and it is not a chatbot. Pair it with `AI_COMPLETE`, or let a Cortex Agent run the whole loop.

Two details that matter more than they look:

- Snowflake recommends chunks of **no more than 512 tokens** (roughly 385 English words) — smaller chunks retrieve better, even with a long-context embedding model available.
- A search service **runs with owner's rights**. Whoever created it decided what is inside it; a querying role needs `USAGE` on the service and on its database and schema, and sees everything the service indexed. Row-level security on the base table is not re-applied at query time.

→ [More on Cortex Search](https://docs.snowflake.com/en/user-guide/snowflake-cortex/cortex-search/cortex-search-overview)

### Choosing between the four

| The data is… | and they want… | Use |
|---|---|---|
| unstructured documents | answers grounded in those documents | **Cortex Search** (+ `AI_COMPLETE`) |
| structured tables | a number, a trend, an aggregate | **Cortex Analyst** |
| both | one conversation across both | **Cortex Agents** |
| both, and nobody wants to build | a ready-made application | **Snowflake CoWork / Intelligence** |

When a question asks which *capability enables* a chatbot over PDFs, the answer is the one doing the retrieval: Cortex Search. An agent would *call* it; CoWork is an application built on agents.

### Where the other unstructured functions fit

| Need | Function |
|---|---|
| PDF / DOCX / PPTX → text or Markdown | `AI_PARSE_DOCUMENT` |
| named fields out of a document | `AI_EXTRACT` |
| audio or video → transcript | `AI_TRANSCRIBE` |
| "are these two things about the same subject?" | `AI_SIMILARITY` |
| licensed third-party corpora with no pipeline to build | **Cortex Knowledge Extensions** |

A Cortex Knowledge Extension is a Cortex Search Service shared through the Snowflake Marketplace or a private listing. The provider loads text into a table, builds a search service on it, and publishes it; you query it like any other service. Providers are told to index a `SOURCE_URL` column so answers can link back to the source. → [More on Cortex Knowledge Extensions](https://docs.snowflake.com/en/user-guide/snowflake-cortex/cortex-knowledge-extensions/cke-overview)

In [ ]:
%%sql -r search_services_list
-- The 1.1-level check: does this account already have a search service to build RAG on?
SHOW CORTEX SEARCH SERVICES IN ACCOUNT;

---

## Check your understanding

Twelve questions on this notebook. Answer before expanding.

**1.** Which two things must a role hold before it can call `AI_COMPLETE`?

<details><summary>Show answer</summary>

The account-level `USE AI FUNCTIONS` privilege (or a per-function `USE AI FUNCTION AI_COMPLETE` grant), **and** one of the database roles `SNOWFLAKE.CORTEX_USER` or `SNOWFLAKE.AI_FUNCTIONS_USER`. Both halves are required, and the failure looks the same either way, so the debugging habit is to check both rather than assume. The tempting wrong answer is "just `CORTEX_USER`" — which is the half that happens to be granted to `PUBLIC` by default, so it is usually the half you already have.

→ [Cortex privileges and access](https://docs.snowflake.com/en/user-guide/snowflake-cortex/aisql-privileges-and-access)

</details>

**2.** What is the default value of `max_tokens` in `AI_COMPLETE`'s `model_parameters`, and what does it limit?

<details><summary>Show answer</summary>

4096, and it caps the **output** only. It does not shrink your prompt or protect you from exceeding a model's context window on the input side — for that you need `AI_COUNT_TOKENS` before the call. A truncated answer with no error is what you get when the output needs more than the cap.

→ [AI_COMPLETE](https://docs.snowflake.com/en/sql-reference/functions/ai_complete-single-string)

</details>

**3.** Which command lists the base models your role can use, and what must you add to it?

<details><summary>Show answer</summary>

`SHOW CORTEX BASE MODELS`, with `IN SCHEMA SNOWFLAKE.MODELS`. Snowflake recommends the clause because without it the command follows ordinary scoping rules and may return zero rows even when the grants are fine — an empty result that reads like "no models" but means "wrong scope". The output is also filtered by your model grants, so two roles running it can legitimately see different lists.

→ [SHOW CORTEX BASE MODELS](https://docs.snowflake.com/en/sql-reference/sql/show-cortex-base-models)

</details>

**4.** A teammate runs `ALTER ACCOUNT SET CORTEX_ENABLED_CROSS_REGION = 'us-east-1';` and it fails. Why?

<details><summary>Show answer</summary>

There is no value naming a single physical region. The documented values stop at cloud plus geography: `ANY_REGION`, the `*_GLOBAL` values, the geography values such as `AWS_US` / `AWS_EU` / `AWS_APJ` / `AWS_JP` / `AWS_AU` / `AZURE_US` / `AZURE_EU` / `GCP_US`, and `DISABLED` — with comma-separated combinations allowed. `'AWS_US'` is the closest legal setting to what they meant. Only `ACCOUNTADMIN` can set this parameter; `ORGADMIN` cannot.

→ [Cross-region inference](https://docs.snowflake.com/en/user-guide/snowflake-cortex/cross-region-inference)

</details>

**5.** A fine-tuning job is created with `SELECT question, answer FROM tickets`. It runs, but the model behaves as if it learned nothing useful. What is wrong?

<details><summary>Show answer</summary>

The training query must return columns named **`prompt`** and **`completion`**. Any other column is ignored, so a query returning `question` and `answer` hands the job no usable training signal. Alias them: `SELECT question AS prompt, answer AS completion FROM tickets`. This is the failure mode that costs money before it shows up — the job consumes credits and then produces a model that has not improved.

→ [Cortex Fine-tuning](https://docs.snowflake.com/en/user-guide/snowflake-cortex/cortex-finetuning)

</details>

**6.** `SELECT SNOWFLAKE.CORTEX.FINETUNE('DESCRIBE', 'MY_DB.PUBLIC.MY_TUNED_MODEL');` — what will this do?

<details><summary>Show answer</summary>

Fail to find anything useful, because `DESCRIBE` takes the **job id** returned by `'CREATE'`, not the model name. The model name is what you pass to `AI_COMPLETE` afterwards; the job id is what you pass to `DESCRIBE` and `CANCEL`. `'SHOW'` takes no argument at all and is the way to find a job id you have lost.

→ [Cortex Fine-tuning](https://docs.snowflake.com/en/user-guide/snowflake-cortex/cortex-finetuning)

</details>

**7.** A team grants `SNOWFLAKE.AI_FUNCTIONS_USER` to their analysts. A dashboard that ran fine last week now fails after someone adds a per-customer summary step. What happened?

<details><summary>Show answer</summary>

`AI_FUNCTIONS_USER` covers the scalar AI functions but deliberately excludes the aggregates `AI_AGG` and `AI_SUMMARIZE_AGG`. The new summarise-per-group step calls one of those, so it is refused while every other function in the query still works. The fix is either `CORTEX_USER` for that role, or rewriting the step without an aggregate AI function — and which you choose is a governance decision, not a technical one.

→ [Cortex privileges and access](https://docs.snowflake.com/en/user-guide/snowflake-cortex/aisql-privileges-and-access)

</details>

**8.** Someone asks you to build "a chatbot that answers questions from our PDF contracts". Which Snowflake capability *enables* it, and what do the other candidates actually do?

<details><summary>Show answer</summary>

**Cortex Search** — it is the piece doing retrieval over unstructured text. Cortex Analyst is text-to-SQL over structured tables via a semantic view, so it is the wrong data type entirely. A Cortex Agent is an orchestrator that would *call* Search. Snowflake CoWork (also written as Snowflake CoWork) is a finished application built on agents. Three of the four are plausible, which is why the question stem's word "capability" is doing the work.

→ [Cortex Search overview](https://docs.snowflake.com/en/user-guide/snowflake-cortex/cortex-search/cortex-search-overview)

</details>

**9.** Your model answers confidently but with facts that are two years out of date. Fine-tune, or retrieve?

<details><summary>Show answer</summary>

Retrieve. Fine-tuning changes *behaviour* — tone, format, vocabulary — and the result still has a fixed cut-off, so every refresh of the facts means another tuning run and another version to govern. Retrieval keeps the facts in a table you can update this afternoon. The cost you accept in exchange is a chunking and indexing pipeline that somebody now maintains, plus per-query retrieval latency. Fine-tuning earns its place when you have labelled `prompt`/`completion` pairs and a format requirement prompting has not met.

→ [Cortex Fine-tuning](https://docs.snowflake.com/en/user-guide/snowflake-cortex/cortex-finetuning)

</details>

**10.** Your account has `CORTEX_USER` on `PUBLIC` and an ungoverned credit bill. What is the trade-off in locking it down?

<details><summary>Show answer</summary>

Revoking `SNOWFLAKE.CORTEX_USER` from `PUBLIC` and granting it to named roles gives you an attributable list of who can spend credits on inference. What you give up is self-service: every new analyst is now blocked until someone grants the role, and queries that worked yesterday fail today with a privilege error that does not name the missing grant. A narrower alternative is to leave `CORTEX_USER` alone and grant `USE AI FUNCTION <name>` per function, which restricts the surface without removing everyone's access at once.

→ [Cortex privileges and access](https://docs.snowflake.com/en/user-guide/snowflake-cortex/aisql-privileges-and-access)

</details>

**11.** You are choosing between a warehouse-served registry model and one served on Snowpark Container Services. What actually decides it?

<details><summary>Show answer</summary>

Packaging and hardware, not accuracy. Warehouse serving is callable straight from SQL, is capped at **15 GB** of model, and uses Snowflake's own package set. SPCS serving means a container: any dependency you like, GPUs if you need them, and a REST endpoint — paid for as compute-pool node-hours that keep running whether or not anyone calls the model. The usual honest answer for a CPU model doing batch scoring is the warehouse; SPCS is what you move to when one of those constraints actually bites. [DD2 — Model Registry](../Deep%20Dives/DD2%20-%20Model%20Registry.ipynb) works through this properly.

→ [Model Registry overview](https://docs.snowflake.com/en/developer-guide/snowflake-ml/model-registry/overview)

</details>

**12.** *(connects to Domain 3 — governance)* You publish a Cortex Search service over HR documents, and grant `USAGE` on it to the analyst role. Row access policies on the underlying table restrict each analyst to their own department. What does an analyst see?

<details><summary>Show answer</summary>

Everything the service indexed. A Cortex Search service performs searches with **owner's rights**: the content was read once, by the owner, at indexing time, and the querying role's own restrictions are not re-applied to the index. The failure this produces is quiet — no error, just an analyst retrieving another department's documents. The fix is at build time: index only what the audience is allowed to see, and build separate services per audience if the boundaries differ.

→ [Cortex Search overview](https://docs.snowflake.com/en/user-guide/snowflake-cortex/cortex-search/cortex-search-overview)

</details>